# Alocação enfermeiro-quarto por turno (NRA)

# Programação Linear Inteira (PLI)

In [ ]:
import pulp
import pandas as pd
import json
import time
import os

def solve_nra_problem(instance_dir):
    # start of time measurement
    start_time = time.perf_counter()

    # data load
    with open(f"{instance_dir}/instance_info.json", "r") as f:
        info = json.load(f)

    nurse_shifts = pd.read_csv(f"{instance_dir}/nurse_shifts.csv")
    room_shifts = pd.read_csv(f"{instance_dir}/occupied_room_shifts.csv")

    # extract penalties weights
    w_skill = info["weights"]["S2_room_nurse_skill"]
    w_work = info["weights"]["S4_nurse_excessive_workload"]

    # init the solver
    prob = pulp.LpProblem("NRA_Optimization", pulp.LpMinimize)

    # binary variables: 1 if the nurse is allocated to the room, otherwise 0
    x_vars = {}  # x[n, r, s]

    # integer variables: nurse workload
    e_vars = {}  # e[n, s]

    objective_terms = []

    # get the each turn
    shifts_unique = room_shifts["global_shift"].unique()

    # for each turn: 0, 1, ..., 41
    for s in shifts_unique:
        rooms_in_shift = room_shifts[room_shifts["global_shift"] == s]
        available_nurses = nurse_shifts[nurse_shifts["global_shift"] == s]

        if available_nurses.empty and not rooms_in_shift.empty:
            continue
        
        # for each nurse
        for _, nurse in available_nurses.iterrows():
            n_id = nurse["nurse_id"]
            n_skill = nurse["skill_level"]
            n_capacity = nurse["max_load"]  

            e_vars[(n_id, s)] = pulp.LpVariable(
                f"e_{n_id}_{s}", lowBound=0, cat=pulp.LpInteger
            )

            # add the penalty weight
            objective_terms.append(w_work * e_vars[(n_id, s)])

            workload_sum = 0

            for _, room in rooms_in_shift.iterrows():
                r_id = room["room_id"]
                r_req_skill = room["max_skill_required"]
                r_workload = room["total_room_workload"]

                # binary variable
                x_vars[(n_id, r_id, s)] = pulp.LpVariable(
                    f"x_{n_id}_{r_id}_{s}", cat=pulp.LpBinary
                )

                # skill deficit
                deficit = max(0, r_req_skill - n_skill)

                # add the deficit
                if deficit > 0:
                    objective_terms.append(w_skill * deficit * x_vars[(n_id, r_id, s)])

                # total workload
                workload_sum += r_workload * x_vars[(n_id, r_id, s)]

            # soft constraint: workload
            prob += (
                workload_sum - n_capacity <= e_vars[(n_id, s)],
                f"Workload_{n_id}_{s}",
            )

        # hard constraint: room coverage 
        for _, room in rooms_in_shift.iterrows():
            r_id = room["room_id"]
            prob += (
                pulp.lpSum(
                    [
                        x_vars[(n["nurse_id"], r_id, s)]
                        for _, n in available_nurses.iterrows()
                    ]
                )
                == 1, # c_1 + c_2 + ... + c_n = 1
                f"Coverage_{r_id}_{s}",
            )

    # calculate all penalties
    prob += pulp.lpSum(objective_terms), "Total_Penalties"

    # solve with CBC
    prob.solve(pulp.PULP_CBC_CMD(msg=True))

    # end time measurement
    end_time = time.perf_counter()
    execution_time = end_time - start_time

    # results
    penalidade_total = pulp.value(prob.objective)
    
    print("\n" + "="*50)
    print("RESULTADOS DO MÉTODO EXATO (PLI - Solver CBC)")
    print("="*50)
    print(f"Status da Resolução                 : {pulp.LpStatus[prob.status]}")
    print(f"Função Objetivo (Penalidade Total)  : {penalidade_total}")
    print(f"Tempo de Processamento              : {execution_time:.2f} segundos")
    print("="*50)

    # save in a json
    ARQUIVO_RESULTADOS = 'resultados_comparativos.json'
    
    if os.path.exists(ARQUIVO_RESULTADOS):
        with open(ARQUIVO_RESULTADOS, 'r', encoding='utf-8') as f:
            dados_json = json.load(f)
    else:
        dados_json = {
            "PLI": {},
            "GA": {
                "penalidade_melhor": 0.0,
                "penalidade_media": 0.0,
                "tempo_medio_segundos": 0.0
            }
        }
    
    # update the PLI keys with the exact results
    dados_json["PLI"]["penalidade_total"] = float(penalidade_total) if penalidade_total is not None else 0.0
    dados_json["PLI"]["tempo_segundos"] = float(execution_time)

    # write back to the file
    with open(ARQUIVO_RESULTADOS, 'w', encoding='utf-8') as f:
        json.dump(dados_json, f, indent=4)

    print(f"-> Arquivo '{ARQUIVO_RESULTADOS}' atualizado com as métricas do PLI!\n")

    return prob